In [1]:
# =======================================================================
#                           IMPORTS FOR DATA PROCESSING
# =======================================================================

# PyMuPDFLoader: A highly efficient PDF document loader often preferred 
# in LangChain for its speed and reliable text extraction from complex PDFs.
from langchain_community.document_loaders import PyMuPDFLoader 

# RecursiveCharacterTextSplitter: Breaks the large document into smaller, 
# manageable pieces ("chunks"). This is crucial for RAG performance as it 
# ensures relevant information fits into the LLM's context window.
from langchain.text_splitter import RecursiveCharacterTextSplitter

c:\Users\hamza\AppData\Local\Programs\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# --- 1. DOCUMENT LOADING AND CHUNKING ---

# Define the file path for the PDF document.
# Using 'r' prefix for raw string to handle backslashes in Windows paths professionally.
# The user's actual path is used here.
FILE_PATH = r"C:\Users\hamza\Downloads\Unpaid_Internship_Offer.pdf" 

# Initialize the PDF Loader (PyMuPDFLoader is highly recommended for stability and speed)
loader = PyMuPDFLoader(FILE_PATH)

# Load the PDF content into LangChain's Document objects.
documents = loader.load()

# Initialize the Text Splitter
# RecursiveCharacterTextSplitter is chosen as it intelligently splits text based on a hierarchy 
# of separators, leading to contextually coherent chunks.
from langchain.text_splitter import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,    # Maximum size of each text chunk (1000 characters)
    chunk_overlap=200   # Overlap between chunks (200 characters) to preserve context
)

# Split the loaded documents into chunks
docs = text_splitter.split_documents(documents)

# --- Verification and Output ---

# Print the total number of chunks created to verify the splitting process.
print(f"Total number of chunks: {len(docs)}")

# Print the content of the first chunk to check the quality of text extraction.
print("\n--- First Chunk Content ---")
print(docs[0].page_content)

Total number of chunks: 3

--- First Chunk Content ---
Internship Offer 
This Remote internship offer is made by CodAgentic, 1608 spruce street Suite 1R 
Philadelphia, PA 19103, to the intern  
Name:  
CNIC:  
Address: 
 
1. Purpose 
The Company agrees to provide the Intern with an unpaid internship to gain practical 
experience related to AI Intern. This internship is educational and does not constitute an 
employment relationship. 
2. Duration 
The internship will begin on SEP 25, 2025, and continue for a period of 12 weeks (Extendable), 
unless terminated earlier by either party with 7 days’ written notice.   
3. Role & Responsibilities 
The Intern will serve as a AI Intern under the guidance of the CodAgentic assigned by the 
Employer. Responsibilities may include but are not limited to: 
• 
Supporting AI-related tasks assigned by the CodAgentic. 
• 
Following company rules and maintaining professional conduct. 
4. Nature of Internship 
• 
This is an unpaid position. No wages, allo

In [3]:
# =======================================================================
#                           STEP 2: EMBEDDINGS AND FAISS STORE
# =======================================================================

# Necessary Imports for this stage
from langchain_community.embeddings import HuggingFaceEmbeddings # For Local Embeddings
from langchain_community.vectorstores import FAISS # For Local Vector Store
import os # To check if the FAISS index already exists

# --- Configuration ---
EMBEDDINGS_MODEL = "all-MiniLM-L6-v2"
FAISS_INDEX_PATH = "faiss_index_local_rag"

# 1. Initialize Local Embeddings Model
# HuggingFaceEmbeddings loads a model from the Sentence-Transformers library locally.
print("\n--- 1. Initializing Local Embeddings Model ---")
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDINGS_MODEL, 
    # model_kwargs explicitly sets the device to CPU. This is best practice 
    # for local deployments to ensure cross-platform compatibility without a GPU.
    model_kwargs={'device': 'cpu'} 
) 

# Verification: Ensure the model is loaded and get the vector dimension.
try:
    if 'docs' in locals() and docs:
        first_chunk_vector = embeddings.embed_query(docs[0].page_content)
        print(f"Local Embeddings Ready. Vector Dimension: {len(first_chunk_vector)}")
    else:
        print("Warning: 'docs' variable not found. Skipping vector dimension check.")
except Exception as e:
    print(f"ERROR: Local embedding model failed to load. Please check packages. ({e})")
    exit()

# 2. Create or Load FAISS Vector Store
print("\n--- 2. Creating/Loading FAISS Vector Store ---")

if os.path.exists(FAISS_INDEX_PATH):
    # Check if the FAISS folder already exists on the disk.
    print(f"Loading existing FAISS index from '{FAISS_INDEX_PATH}'...")
    vectorstore = FAISS.load_local(
        FAISS_INDEX_PATH, 
        embeddings, 
        allow_dangerous_deserialization=True # Required for newer versions of LangChain
    )
    print("FAISS Store Loaded successfully.")
else:
    # If the index doesn't exist, create a new one from the document chunks.
    print("🛠️ Creating new FAISS Vector Store from chunks...")
    vectorstore = FAISS.from_documents(docs, embeddings)
    
    # Save the new store locally for persistence and reuse.
    vectorstore.save_local(FAISS_INDEX_PATH)
    print(f"New Local FAISS Store Created & Saved.")

print(f"Vector store is ready at: '{FAISS_INDEX_PATH}'.")


--- 1. Initializing Local Embeddings Model ---


C:\Users\hamza\AppData\Local\Temp\ipykernel_7332\647618400.py:17: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x000001D4E9A6EE10>: Failed to resolve \'huggingface.co\' ([Errno 11001] getaddrinfo failed)"))'), '(Request ID: 8d16c1bb-2c5f-454e-b6f8-2b4484b380d0)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/./modules.json
Retrying in 1s [Retry 1/5].
'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x000001D4E9A4ED50>: Failed to resolve \'huggingface.co\' ([Errno 11001] getaddrinfo failed)"))'), '(Request ID: d6041d4b-0da4-401f-90fe-e33fb12329c2)')' thrown while requesting HEAD https://huggingface.

Local Embeddings Ready. Vector Dimension: 384

--- 2. Creating/Loading FAISS Vector Store ---
Loading existing FAISS index from 'faiss_index_local_rag'...
FAISS Store Loaded successfully.
Vector store is ready at: 'faiss_index_local_rag'.


In [4]:
# =======================================================================
#                      PART A: RAG SYSTEM SETUP
# =======================================================================
# This section initializes:
#    Embeddings Model (HuggingFace)
#    FAISS Vector Store (local)
#    Ollama LLM (cloud model)
#    Conversation Memory + RAG Chain
# =======================================================================

from langchain_community.llms import Ollama
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferMemory
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

# --- Configuration ---
EMBEDDINGS_MODEL = "all-MiniLM-L6-v2"
FAISS_INDEX_PATH = "faiss_index_local_rag"
LLM_MODEL = "gpt-oss:120b-cloud"

# =======================================================================
#                           LOAD VECTOR STORE
# =======================================================================

print("\n--- Loading Local Vector Store ---")

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDINGS_MODEL,
    model_kwargs={'device': 'cpu'}
)

try:
    vectorstore = FAISS.load_local(
        FAISS_INDEX_PATH,
        embeddings,
        allow_dangerous_deserialization=True
    )
    print(f"Local FAISS Store '{FAISS_INDEX_PATH}' loaded successfully.")
except Exception as e:
    print(f"FATAL ERROR: Failed to load Vector Store file. ({e})")
    vectorstore = None

# =======================================================================
#                           INITIALIZE LLM
# =======================================================================

print("\n--- Connecting to Ollama Cloud Model ---")
try:
    llm = Ollama(model=LLM_MODEL)
    print(f"LLM '{LLM_MODEL}' initialized successfully.")
except Exception as e:
    print(f"ERROR: Failed to initialize LLM ({e})")
    llm = None

# =======================================================================
#                           MEMORY + RAG CHAIN
# =======================================================================

if llm and vectorstore:
    memory = ConversationBufferMemory(
        memory_key="chat_history",
        return_messages=True,
        output_key='answer'
    )

    qa_chain = ConversationalRetrievalChain.from_llm(
        llm=llm,
        retriever=vectorstore.as_retriever(),
        memory=memory,
        return_source_documents=True,  #  Keep True (for internal use)
        output_key='answer'
    )

    print("\nRAG Chain Setup Complete. Ready to Query!")
else:
    print("\nERROR: RAG Chain not created (missing LLM or VectorStore).")
    qa_chain = None



--- Loading Local Vector Store ---
Local FAISS Store 'faiss_index_local_rag' loaded successfully.

--- Connecting to Ollama Cloud Model ---
LLM 'gpt-oss:120b-cloud' initialized successfully.

RAG Chain Setup Complete. Ready to Query!


C:\Users\hamza\AppData\Local\Temp\ipykernel_7332\2454721943.py:50: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaLLM``.
  llm = Ollama(model=LLM_MODEL)
C:\Users\hamza\AppData\Local\Temp\ipykernel_7332\2454721943.py:61: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(


In [ ]:
# =======================================================================
#                         PART B: RAG CHAT INTERFACE
# =======================================================================
# Features:
#    GPT-like minimal clean layout
#    Chat bubbles for user & bot
#    Integrated with Local RAG Chain (Ollama Cloud)
#    Responsive layout (Textbox + Send + Clear)
# =======================================================================

import gradio as gr

# =======================================================================
#                             CHAT FUNCTION
# =======================================================================
def chat_interface(user_input, history):
    """Handles user query and returns chatbot response (only the answer)."""
    print(f"\n User query received: {user_input}")

    if qa_chain is None:
        error_message = "Error: RAG chain not initialized properly."
        history = history or []
        history.append([user_input, error_message])
        return "", history

    try:
        response = qa_chain({"question": user_input})
        answer = response.get('answer', ' No response from model.').strip()
        history = history or []
        history.append([user_input, answer])
        print("RAG Response:", answer[:150], "...")
        return "", history
    except Exception as e:
        error_message = f"Error: {type(e).__name__} - {e}"
        history.append([user_input, error_message])
        print(error_message)
        return "", history

# =======================================================================
#                            CLEAR FUNCTION
# =======================================================================
def clear_chat():
    """Clears chat history and memory."""
    try:
        if memory:
            memory.clear()
            print("Memory cleared successfully.")
    except Exception as e:
        print(f"Failed to clear memory: {e}")
    return "", []

# =======================================================================
#                        CUSTOM THEME + CSS
# =======================================================================
custom_css = """
body {
    background-color: #f5f7fb;
}
.gradio-container {
    font-family: 'Inter', sans-serif;
}
#chatbot {
    height: 550px !important;
    border-radius: 16px;
    background-color: #ffffff;
    box-shadow: 0px 4px 14px rgba(0,0,0,0.08);
    padding: 10px;
}

/*  Chat bubbles */
.message.user {
    background-color: #e8f0fe !important;
    color: #1a1a1a !important;
    border-radius: 16px 16px 0 16px !important;
    padding: 10px 14px !important;
}
.message.bot {
    background-color: #f0f0f0 !important;
    color: #111111 !important;
    border-radius: 16px 16px 16px 0 !important;
    padding: 10px 14px !important;
}

/*  Input row layout (GPT-style) */
#input-row {
    display: flex;
    align-items: center;
    justify-content: center;
    gap: 6px;
    margin-top: 12px;
}

/*  Textbox = 92%, Send = 4%, Clear = 4% */
#msg-box {
    flex: 0 0 92%;
    min-height: 92px !important; /*  increased height by 10% */
    border-radius: 12px !important;
    border: 1px solid #d0d7de !important;
    padding-left: 10px !important;
}

/*  Buttons adjusted */
#send-btn {
    flex: 0 0 3.5%; /*  slightly reduced width (was 4%) */
    width: 70px !important;
    height: 70px !important;
    font-size: 14px !important;
    border-radius: 12px !important;
}

/*  Clear button — red color & smaller size */
#clear-btn {
    flex: 0 0 3.5%; /*  same smaller width */
    width: 70px !important;
    height: 70px !important;
    font-size: 14px !important;
    border-radius: 12px !important;
    background-color: #ff4d4d !important; /*  red color */
    color: white !important;
    border: none !important;
}

footer {
    text-align: center;
    font-size: 14px;
    color: gray;
    margin-top: 10px;
}
"""

theme = gr.themes.Soft(
    primary_hue="sky",
    secondary_hue="gray",
    text_size="lg",
    spacing_size="lg"
).set()

# =======================================================================
#                        GRADIO UI LAYOUT
# =======================================================================
with gr.Blocks(theme=theme, css=custom_css, title=" RAG Chatbot") as demo:
    gr.Markdown("<h1 style='text-align:center;'> RAG Chatbot</h1>")
    gr.Markdown("<p style='text-align:center;'>Ask anything about your local documents <br><b>Model:</b> gpt-oss:120b-cloud</p>")

    chatbot = gr.Chatbot(label=" Chat History", elem_id="chatbot", show_copy_button=True)

    with gr.Row(elem_id="input-row"):
        msg = gr.Textbox(
            placeholder="Type your message here...",
            show_label=False,
            elem_id="msg-box",
        )
        send_btn = gr.Button("⬆️", variant="primary", elem_id="send-btn")
        clear_btn = gr.Button("🗑️", elem_id="clear-btn")

    # Bind functions
    msg.submit(chat_interface, [msg, chatbot], [msg, chatbot])
    send_btn.click(chat_interface, [msg, chatbot], [msg, chatbot])
    clear_btn.click(clear_chat, None, [msg, chatbot], queue=False)

    gr.Markdown("<footer> <b>Tip:</b> Response time may vary depending on Ollama Cloud performance.</footer>")

# =======================================================================
#                           LAUNCH APP
# =======================================================================
if __name__ == "__main__":
    demo.launch(inbrowser=True, debug=False)


C:\Users\hamza\AppData\Local\Temp\ipykernel_7332\2803551855.py:145: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(label=" Chat History", elem_id="chatbot", show_copy_button=True)


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.



 User query received: what is the main purpose of this internship ?


C:\Users\hamza\AppData\Local\Temp\ipykernel_7332\2803551855.py:27: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  response = qa_chain({"question": user_input})


RAG Response: The main purpose of the internship is for the Company to give the Intern an unpaid, educational experience that provides practical, hands‑on training  ...

 User query received: what are the main responsbilities of an AI intern at CodeAgentic according to the offer?
RAG Response: According to the internship offer, the AI Intern’s main responsibilities are:

1. **Supporting AI‑related tasks** that are assigned by CodAgentic.  
2 ...

 User query received: Under what conditions can the internship agreement be terminated at CodeAgentic?
RAG Response: CodeAgentic (the Company) can end the internship in two ways:

1. **Standard termination** – Either the Company or the intern may terminate the agreem ...
Memory cleared successfully.

 User query received: I am going to skip this internship 
RAG Response: If you’ve decided not to take the internship, the best approach is to let CodAgentic know as soon as possible and to do it in writing. Here’s a simple ...
